# LAB | Abstractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
# Restart the kernel and run this cell before all other imports.

# Use the project GPU kernel locally; restart after installing packages.
%pip install -q torch==2.9.0 torchvision==0.24.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -q datasets==2.21.0 huggingface_hub==0.25.2 pinecone-client==3.1.0 sentence-transformers==3.0.1 transformers==4.44.2 pandas tqdm python-dotenv

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\zinam\OneDrive\Desktop\lab-abstractive-question-answering\.venv\Scripts\python.exe -m pip install --upgrade pip


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [2]:
from datasets import load_dataset

# We use the "wiki_snippets" dataset, which contains millions of short passages
# extracted from Wikipedia. Each passage comes with useful metadata: the article
# it was taken from (article_title), the section within that article (section_title),
# and the passage's text itself (passage_text). We use the "wiki40b_en_100_0"
# configuration, which contains English passages of ~100 words each.
wiki_data = load_dataset(
    'community-datasets/wiki_snippets',
    name='wiki40b_en_100_0',
    split='train',
    streaming=True
).shuffle(seed=960)

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [3]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'_id': '{"datasets_id": 1251236, "wiki_id": "Q2049363", "sp": 16, "sc": 56, "ep": 16, "ec": 109}',
 'datasets_id': 1251236,
 'wiki_id': 'Q2049363',
 'start_paragraph': 16,
 'start_character': 56,
 'end_paragraph': 16,
 'end_character': 109,
 'article_title': 'Panemotichus',
 'section_title': 'Site',
 'passage_text': 'Archaeologists have revealed Iron Age remains there.'}

In [4]:
# Keep passages whose section title contains "History" (case insensitive).
history = wiki_data.filter(
    lambda document: 'history' in (document['section_title'] or '').lower()
)

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [5]:
from itertools import islice
from tqdm.auto import tqdm

# Reduce this value for a quick trial; the full lab uses 50,000 passages.
total_doc_count = 50_000
docs = []
for document in tqdm(islice(history, total_doc_count), total=total_doc_count):
    docs.append({
        'article_title': document['article_title'],
        'section_title': document['section_title'],
        'passage_text': document['passage_text'],
    })
counter = len(docs)
print(f'Collected {counter:,} historical passages.')

  0%|          | 0/50000 [00:00<?, ?it/s]

Collected 50,000 historical passages.


In [6]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,Persian spider gecko,Natural history & Sexing,"'eyelid'"" is an adaptation to diurnal life. Th..."
1,Paffendorf,History,Paffendorf History Ramon Zenker was involved i...
2,Peranakan,History,the Chinese was widespread.\nIt cannot be deni...
3,Pennsylvania Route 244,History,244 to its original alignment.
4,Peretz (Russian TV channel),History of channel,Peretz (Russian TV channel) History of channel...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [7]:
import os
import getpass
from pinecone import Pinecone
from dotenv import load_dotenv

# Load local credentials without displaying them or overriding environment values.
load_dotenv()

#
# For this notebook, you'll need a Pinecone API key, 
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager 
# (the key icon in the left sidebar). 
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [8]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [9]:
index_name = 'abstractive-question-answering'

In [10]:
import time

if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=768, metric='cosine', spec=spec)

deadline = time.monotonic() + 300
while True:
    description = pc.describe_index(index_name)
    if description.dimension != 768 or description.metric != 'cosine':
        raise ValueError('The existing index must use 768 dimensions and cosine similarity.')
    if description.status['ready']:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError('Pinecone index was not ready after five minutes.')
    time.sleep(2)

index = pc.Index(index_name)
stats = index.describe_index_stats()
print(stats)
# A newly created index is empty. Reusing this index preserves existing vectors;
# the deterministic IDs below overwrite the same passages on a repeated run.

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 10688}},
 'total_vector_count': 10688}


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [11]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
retriever = SentenceTransformer(
    'flax-sentence-embeddings/all_datasets_v3_mpnet-base', device=device
)
assert retriever.get_sentence_embedding_dimension() == 768
retriever

Using: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


C:\Users\zinam\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: MPNetModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [ ]:
import hashlib

batch_size = 64
for i in tqdm(range(0, len(df), batch_size)):
    end = min(i + batch_size, len(df))
    batch = df.iloc[i:end]
    metadata = batch.to_dict(orient='records')
    embeddings = retriever.encode(
        batch['passage_text'].tolist(), batch_size=batch_size,
        normalize_embeddings=True, show_progress_bar=False,
    ).tolist()
    ids = [
        hashlib.sha256(
            (m['article_title'] + '\n' + m['section_title'] + '\n' + m['passage_text']).encode('utf-8')
        ).hexdigest()
        for m in metadata
    ]
    index.upsert(vectors=[
        {'id': identifier, 'values': embedding, 'metadata': meta}
        for identifier, embedding, meta in zip(ids, embeddings, metadata)
    ])

# Pinecone is eventually consistent; the count may take time to update.
index.describe_index_stats()

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [ ]:
def query_pinecone(query, top_k):
    if not query.strip() or top_k < 1:
        raise ValueError('Provide a nonempty question and a positive top_k.')
    xq = retriever.encode(query, normalize_embeddings=True).tolist()
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return xc

In [ ]:
def format_query(query, context):
    passages = [f"<P> {m['metadata']['passage_text']}" for m in context]
    if not passages:
        raise ValueError('No context was retrieved for this question.')
    context = ' '.join(passages)
    return f'question: {query} context: {context}'

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [ ]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

In [ ]:
from pprint import pprint

In [ ]:
# format the query in the form generator expects the input
query = format_query(query, result["matches"])
pprint(query)

The output looks great. Now let's write a function to generate answers.

In [ ]:
def generate_answer(query):
    inputs = tokenizer(
        [query], max_length=1024, truncation=True, return_tensors='pt'
    ).to(device)
    generator.eval()
    with torch.inference_mode():
        ids = generator.generate(
            **inputs, num_beams=2, min_length=20, max_length=100,
            do_sample=False,
        )
    return tokenizer.batch_decode(
        ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

In [ ]:
generate_answer(query)

As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [ ]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [ ]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [ ]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

In [ ]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

Let’s finish with a final few questions.

In [ ]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

In [ ]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

In [ ]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

<br>

### 🚀 Bonus Challenges

Try one or more of these to practice:

- Ask your own questions and compare answers using different `top_k` values.
- Ask a question the dataset likely can't answer and observe how the model responds.
- Change the `section_title` filter to a topic other than "History" and rebuild a small index.
- Print the retrieved passages next to each generated answer to see how retrieval quality affects the result.
- Swap the retriever or generator for a different model from Hugging Face and compare outputs.

In [ ]:
# Bonus: compare retrieval depth and inspect the evidence for each answer.
question = 'How did the war of currents change electricity distribution?'
for top_k in (1, 3, 5):
    matches = query_pinecone(question, top_k=top_k)['matches']
    print(f'\ntop_k={top_k}')
    for match in matches:
        print(f"Score: {match['score']:.3f} | {match['metadata']['article_title']}")
        print(match['metadata']['passage_text'])
    print('Answer:', generate_answer(format_query(question, matches)))
# Compare whether the answer is supported by these passages. More passages can
# improve coverage, but irrelevant text or the 1,024-token limit can hurt it.

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [ ]:
# To delete the Pinecone index, 
# uncomment the line below + run this code cell

# pc.delete_index(index_name)